## Convolutional Neural Networks & Recurrent Neural Network 

Membahas photo per frame dan per pixel, penggunaan matriks di CNN dan lainnya<br>
CNN cukup bagus untuk menangkap ekstraksi feature<br>
RNN -> LSTM -> Transformer<br>
fillter itu sama dengan fizze/ kernel

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([          # sama seperti pipeline
    transforms.ToTensor(),                # mengubah data kita dari numpy menjadi tensor [[]], lalu di normalisasi
    transforms.Normalize((0.5,), (0.5,))  # output -> seperti minmax scaler dari rentang -1, 1
])      # 0.5 = mean, 0.5 = std

In [ ]:
# datanya sudah di train dan test, lalu sudah di transform juga
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
# akan membuat folder data yg menyimpan MNIST

100%|██████████| 9.91M/9.91M [00:51<00:00, 191kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 110kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 644kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.15MB/s]


# Searching aja apa itu Batch Size dan apa kegunaannya

In [6]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)   # batch size, yg data kita 1.000 dibagi 64, 64, 64. ini bikin mudah dan ringan
test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [ ]:
# penjelasan di 1.28
cnn_model = nn.Sequential(          # kita set fillter nya 32
    nn.Conv2d(1, 32, kernel_size=3, padding=1),  # output: [32, 28, 28]     # 32 fillter
    nn.ReLU(),
    nn.MaxPool2d(2, 2),                          # output: [32, 14, 14]     # gunakan apply 2x2

    nn.Conv2d(32, 64, kernel_size=3, padding=1), # output: [64, 14, 14]
    nn.ReLU(),
    nn.MaxPool2d(2, 2),                          # output: [64, 7, 7]

    nn.Flatten(),               # ubah representasi gambar ini menjadi vektor, lalu dimasukan ke neural network kita
    nn.Linear(64*7*7, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
# searching saja apa itu Conv2d di GPT 

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model = cnn_model.to(device)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [9]:
epochs = 3
for epoch in range(epochs):
    cnn_model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.1481
Epoch 2, Loss: 0.0442
Epoch 3, Loss: 0.0304


In [10]:
cnn_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cnn_model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 98.41%


sedikit sulit di pahami untuk CNN ini, maaf masih pemula